<a href="https://colab.research.google.com/github/whiteraay/research/blob/main/colab/01_operator_workspace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APD benchmark — Operator workspace

You run the code; the chat model (Claude / ChatGPT / Gemini, free web chat) writes it. Data stays on Drive, mounted read-only. Your outputs are saved to `MyDrive/apd_outputs_<model>/` and survive Colab resets.

**Every session:** run cell 1. **New chat:** cell 2 (first time) or cell 4 (resume). **Then loop:** paste the model's cells into the empty cells at the bottom, run, copy the part between `COPY FROM HERE` and `END OF COPY` back into the chat.

**Before the first session**
- Drive: open the shared **`apd_benchmark`** folder → ⋮ → Organize → **Add shortcut** → My Drive.
- Colab: Tools → Settings → turn **off** AI-powered code completions / generation. Never use Colab's built-in Gemini panel. The code must come only from your chat.
- Your chat: turn off "use my conversations to improve the model" (or equivalent).

In [4]:
#@title 1. Setup — run at the start of EVERY Colab session
# Mount Google Drive (NOT upload) and get the project code.
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob, shutil, subprocess
REPO = "/content/research"
if not os.path.exists(REPO):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # optional Colab secret (key icon on the left)
    except Exception:
        pass
    if token:
        subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/whiteraay/research.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/whiteraay/research.git"])
        print("code: cloned from GitHub")
    else:
        cands = ["/content/drive/MyDrive/apd_benchmark/code/research",
                 *glob.glob("/content/drive/.shortcut-targets-by-id/*/apd_benchmark/code/research")]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            raise SystemExit("No GITHUB_TOKEN secret and no code copy at MyDrive/apd_benchmark/code/research. "
                             "Add the apd_benchmark shortcut to My Drive (operators) or add a GITHUB_TOKEN secret.")
        shutil.copytree(src, REPO)
        print("code: copied from", src)
sys.path.insert(0, REPO + "/src")
from apd_bench import colab as apd

MODEL = "claude"     # <- operator 1: "claude" | operator 2: "gpt" | operator 3: "gemini"
apd.setup_workspace(MODEL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workspace ready for claude.
  model         claude
  repo          /content/research
  data          /content/drive/MyDrive/apd_benchmark/data
  outputs       /content/drive/MyDrive/apd_outputs_claude
  train_videos  146
  test_videos   62
  audit         True
  state         False


{'model': 'claude',
 'repo': '/content/research',
 'data': '/content/drive/MyDrive/apd_benchmark/data',
 'outputs': '/content/drive/MyDrive/apd_outputs_claude',
 'train_videos': 146,
 'test_videos': 62,
 'audit': True,
 'state': False}

## 2. First chat message: charter + execution protocol
Copy everything between the markers into a **new** chat. Wait for the model's acknowledgement.

In [6]:
#@title 2. Print charter + protocol
apd.prompt(0)

==================== COPY FROM HERE INTO THE CHAT ====================
# PROJECT CHARTER — read fully before doing anything

## Your role
You are the lead data scientist on a clinical machine-learning study. You will design, implement, evaluate and document a complete method for identifying participants at elevated acute psychological distress risk from video-recorded psychological interviews. You decide the methodology: which features, which pre-trained models, which classifier, how to fuse channels. Nobody will tell you what to use; your choices and the reasons for them are part of what is being studied.

Two other AI systems are receiving this same charter, the same data, the same compute and the same human operator protocol. Their work and yours will be compared on identical held-out sessions. Make the choices you believe are scientifically best, not the choices you think a comparison wants to see.

## How you work in this study (important)
You cannot run code and you cannot see th

## 3. Phase prompts
Send the next phase only after the model says **PHASE N COMPLETE** and `apd.ls()` shows its files.

In [7]:
#@title 3. Print a phase prompt
PHASE = 1   # 1 audit · 2 proposal · 3 build · 4 evaluate · 5 affect · 6 report
apd.prompt(PHASE)

==================== COPY FROM HERE INTO THE CHAT ====================
# PHASE 1 — DATA AUDIT

Before proposing any method, understand the data — through code, since you cannot look at it. Deliverable: `outputs/claude/audit_notes.md` (plus the small scripts you used, under `src/audit/`). Do not extract model features or train anything in this phase.

1. **Inventory.** Write a script that prints: sessions per cohort (train) and in total (test); participants per cohort; sessions per participant; file formats, resolution, frame rate; duration distribution (min, quartiles, max) for train and test.
2. **Recording conditions.** Read `data/audit.csv`. For duration_s, mean_luminance, face_detect_rate and audio_snr_db print the distribution, and in the training set test whether each differs between cohorts (effect size + a simple test). Also fit a cross-validated (grouped by participant) logistic regression on audit.csv columns alone and print its AUROC: this is the **confound floor** every lat

## 4. Usage limit hit / chat broke / new day
Open a **new** chat with the **same** model and paste this one message. It contains the charter, the protocol, STATE.md, decisions.md and the file list.

In [ ]:
#@title 4. Print resume packet
apd.resume_packet()

## 5. Log what happens in the chat
Record the exact model name the chat shows, and every limit, downgrade, refusal or new chat. These are part of the results.

In [9]:
#@title 5. Log a chat event
EVENT = "start"        # start | limit | resume | downgrade | refusal | error | phase_done | gpu_on | gpu_unavailable
MODEL_SHOWN = "Sonnet 5.5"       # exactly as the chat UI shows it, e.g. the model picker label
DETAIL = ""
apd.log_chat(EVENT, MODEL_SHOWN, DETAIL)

logged: chat start Sonnet 5.5


## 6. Paste the model's cells below
One model cell per Colab cell, in order, exactly as written. Add more cells with **+ Code**. Then paste the printed summary back into the chat.

In [10]:
%%writefile outputs/claude/requirements.txt
numpy
pandas
scipy
scikit-learn
librosa
soundfile
faster-whisper

Overwriting outputs/claude/requirements.txt


In [11]:
apd.run("pip install -q -r outputs/claude/requirements.txt && python -c \"import numpy,pandas,scipy,sklearn,librosa,soundfile,faster_whisper,cv2;print('numpy',numpy.__version__,'pandas',pandas.__version__,'scipy',scipy.__version__,'sklearn',sklearn.__version__,'librosa',librosa.__version__,'soundfile',soundfile.__version__,'faster_whisper',faster_whisper.__version__,'cv2',cv2.__version__)\"")

numpy 2.4.6 pandas 2.2.3 scipy 1.16.3 sklearn 1.6.1 librosa 0.11.0 soundfile 0.14.0 faster_whisper 1.2.1 cv2 5.0.0

==================== COPY FROM HERE INTO THE CHAT ====================
$ pip install -q -r outputs/claude/requirements.txt && python -c "import numpy,pandas,scipy,sklearn,librosa,soundfile,faster_whisper,cv2;print('numpy',numpy.__version__,'pandas',pandas.__version__,'scipy',scipy.__version__,'sklearn',sklearn.__version__,'librosa',librosa.__version__,'soundfile',soundfile.__version__,'faster_whisper',faster_whisper.__version__,'cv2',cv2.__version__)"
[exit code 0, 15s, 1 lines; full log saved as outputs/claude/logs/cmd_0001.log]
numpy 2.4.6 pandas 2.2.3 scipy 1.16.3 sklearn 1.6.1 librosa 0.11.0 soundfile 0.14.0 faster_whisper 1.2.1 cv2 5.0.0
==================== END OF COPY ====================


0

In [12]:
# CELL from the model
%%writefile outputs/claude/src/audit/common.py
import os
import numpy as np
import pandas as pd

VID_EXT = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".mpg", ".mpeg", ".wmv", ".flv")
AUDIT_DIR = "outputs/claude/cache/audit"


def norm_id(x):
    return os.path.splitext(os.path.basename(str(x)))[0]


def list_videos(root):
    out = []
    for d, _, fs in os.walk(root):
        for f in sorted(fs):
            if f.lower().endswith(VID_EXT):
                out.append(os.path.join(d, f))
    return sorted(out)


def load_all():
    """Returns (labels, audit, files). All keyed by 'sid' = filename stem."""
    lab = pd.read_csv("data/train/train_labels.csv")
    lab["sid"] = lab["session_id"].map(norm_id)
    lab["cohort"] = lab["cohort"].astype(str).str.strip().str.capitalize()
    aud = pd.read_csv("data/audit.csv")
    key = next((c for c in ["session_id", "video", "filename", "file", "path", "video_path"]
                if c in aud.columns), aud.columns[0])
    aud["sid"] = aud[key].map(norm_id)
    files = list_videos("data/train") + list_videos("data/test")
    ft = pd.DataFrame({"path": files})
    ft["sid"] = ft["path"].map(norm_id)
    ft["split"] = np.where(ft["path"].str.startswith("data/train"), "train", "test")
    ft["ext"] = ft["path"].map(lambda p: os.path.splitext(p)[1].lower())
    ft["folder_cohort"] = ft["path"].map(
        lambda p: "Experimental" if "/Experimental/" in p else ("Control" if "/Control/" in p else ""))
    return lab, aud, ft

Writing outputs/claude/src/audit/common.py


In [13]:
# CELL from the model
%%writefile outputs/claude/src/audit/inventory.py
"""Phase 1 items 1, 2, 5: inventory, recording conditions, confound floor, strata."""
import os, sys, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from common import load_all, AUDIT_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
SEED = 0
os.makedirs(AUDIT_DIR, exist_ok=True)


def q(s):
    return s.dropna().astype(float).quantile([0, .25, .5, .75, 1]).round(2).tolist()


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj


lab, aud, ft = load_all()

print("=== 1. INVENTORY ===")
print("audit.csv columns:", list(aud.columns))
print("files by split/ext:\n", ft.groupby(["split", "ext"]).size().to_string())
print("label rows:", len(lab), "| unique sid in labels:", lab.sid.nunique(),
      "| duplicate stems among all files:", int(ft.sid.duplicated().sum()))
print("labels without file:", int((~lab.sid.isin(ft.sid)).sum()),
      "| train files without label:", int((~ft[ft.split == "train"].sid.isin(lab.sid)).sum()),
      "| audit rows without file:", int((~aud.sid.isin(ft.sid)).sum()),
      "| files without audit row:", int((~ft.sid.isin(aud.sid)).sum()))
chk = ft[ft.split == "train"].merge(lab[["sid", "cohort"]], on="sid", how="inner")
print("folder-vs-csv cohort disagreements:", int((chk.folder_cohort != chk.cohort).sum()))

print("\nTrain sessions per cohort:\n", lab.cohort.value_counts().to_string())
print("Test sessions (files):", int((ft.split == "test").sum()))
pc = lab.groupby("cohort").participant_id.nunique()
print("Participants per cohort:\n", pc.to_string())
both = lab.groupby("participant_id").cohort.nunique()
print("Participants appearing in both cohorts:", int((both > 1).sum()), "of", lab.participant_id.nunique())
spp = lab.groupby("participant_id").size()
print("Sessions per participant (count of participants by n sessions):\n", spp.value_counts().sort_index().to_string())
print("Sessions per participant by cohort (mean, max):\n",
      lab.groupby(["cohort", "participant_id"]).size().groupby("cohort").agg(["mean", "max"]).round(2).to_string())

# merge audit with split / labels
df = aud.merge(ft[["sid", "split"]].drop_duplicates("sid"), on="sid", how="left")
df = df.merge(lab[["sid", "cohort", "participant_id"]], on="sid", how="left")
df.to_csv(f"{AUDIT_DIR}/merged.csv", index=False)
tr = df[df.split == "train"].dropna(subset=["cohort"]).copy()
te = df[df.split == "test"].copy()
tr["y"] = (tr.cohort == "Experimental").astype(int)
y = tr.y.values
g = tr.participant_id.values
print(f"\nMerged: train rows {len(tr)}, test rows {len(te)}")

for c in ["width", "height"]:
    pass
if {"width", "height"} <= set(df.columns):
    df["res"] = df.width.astype(int).astype(str) + "x" + df.height.astype(int).astype(str)
    print("Resolution (top 8) train:\n", df[df.split == "train"].res.value_counts().head(8).to_string())
    print("Resolution (top 8) test:\n", df[df.split == "test"].res.value_counts().head(8).to_string())
if "fps" in df.columns:
    print("fps rounded (train / test):\n",
          pd.concat([df[df.split == "train"].fps.round(0).value_counts().rename("train"),
                     df[df.split == "test"].fps.round(0).value_counts().rename("test")], axis=1).fillna(0).astype(int).head(8).to_string())
print("duration_s quantiles [min,q1,med,q3,max]  train:", q(tr.duration_s), " test:", q(te.duration_s))
for c in ["Experimental", "Control"]:
    print("  train", c, q(tr[tr.cohort == c].duration_s))

print("\n=== 2. RECORDING CONDITIONS vs COHORT (train) ===")
COND = [c for c in ["duration_s", "mean_luminance", "luminance_std", "face_detect_rate",
                    "audio_snr_db", "speech_fraction", "audio_rms_db"] if c in tr.columns]
rows = []
for c in COND:
    x = tr[c].astype(float)
    m = x.notna().values
    a, b = x[m & (y == 1)], x[m & (y == 0)]
    sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    d = (a.mean() - b.mean()) / sd if sd > 0 else np.nan
    p_s = stats.mannwhitneyu(a, b).pvalue
    auc = roc_auc_score(y[m], x[m])
    pm = tr.groupby("participant_id").agg(v=(c, "mean"), yy=("y", "max")).dropna()
    p_p = stats.mannwhitneyu(pm.v[pm.yy == 1], pm.v[pm.yy == 0]).pvalue
    rows.append(dict(var=c, n_missing=int((~m).sum()), q_Exp=q(a), q_Ctl=q(b), cohen_d=d, auroc_uni=auc,
                     p_session=p_s, p_participant=p_p))
res = pd.DataFrame(rows)
res["p_part_holm"] = holm(res.p_participant.values)
print("Quantile lists are [min,q1,med,q3,max]. p_session ignores clustering (anti-conservative); p_participant uses per-participant means.")
print(res[["var", "n_missing", "cohen_d", "auroc_uni", "p_session", "p_participant", "p_part_holm"]].round(4).to_string(index=False))
for _, r in res.iterrows():
    print(f"  {r['var']}: Exp {r.q_Exp} | Ctl {r.q_Ctl}")
res.to_csv(f"{AUDIT_DIR}/cond_vs_cohort.csv", index=False)

print("\nTrain vs test distribution shift (median train / median test / MWU p):")
for c in COND:
    a, b = tr[c].dropna(), te[c].dropna()
    if len(b):
        print(f"  {c}: {a.median():.3f} / {b.median():.3f} / p={stats.mannwhitneyu(a, b).pvalue:.3g}")

print("\n--- Confound floor: participant-grouped CV on audit.csv columns only ---")
tr["log_duration"] = np.log1p(tr.duration_s)
ALL = [c for c in ["log_duration", "width", "height", "fps", "mean_luminance", "luminance_std",
                   "face_detect_rate", "audio_snr_db", "speech_fraction", "audio_rms_db"] if c in tr.columns]
SETS = {"all_audit": ALL, "duration_only": ["log_duration"],
        "luminance_only": [c for c in ["mean_luminance", "luminance_std"] if c in tr.columns],
        "audio_only": [c for c in ["audio_snr_db", "audio_rms_db", "speech_fraction"] if c in tr.columns],
        "face_rate_only": [c for c in ["face_detect_rate"] if c in tr.columns],
        "format_only": [c for c in ["width", "height", "fps"] if c in tr.columns]}
n_part_min = int(pc.min())
NS = max(2, min(5, n_part_min))
print("folds:", NS, "| repeated over 5 seeds")


def lr():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))


def gb():
    return make_pipeline(SimpleImputer(strategy="median"),
                         GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, random_state=SEED))


def cv_auc(X, yy, gg, mk, seed):
    oof = np.zeros(len(yy))
    for a, b in StratifiedGroupKFold(n_splits=NS, shuffle=True, random_state=seed).split(X, yy, gg):
        m = mk()
        m.fit(X[a], yy[a])
        oof[b] = m.predict_proba(X[b])[:, 1]
    return roc_auc_score(yy, oof)


out = []
for name, cols in SETS.items():
    if not cols:
        continue
    X = tr[cols].astype(float).values
    for mname, mk in [("logreg", lr), ("gbm_depth2", gb)]:
        a = [cv_auc(X, y, g, mk, s) for s in range(5)]
        out.append(dict(features=name, model=mname, auroc_mean=np.mean(a), auroc_sd=np.std(a), n_features=len(cols)))
fl = pd.DataFrame(out)
print(fl.round(3).to_string(index=False))
fl.to_csv(f"{AUDIT_DIR}/confound_floor.csv", index=False)

# permutation null at participant level (only meaningful if participants are single-cohort)
rng = np.random.RandomState(SEED)
pl = tr.groupby("participant_id").y.max()
X = tr[ALL].astype(float).values
obs = np.mean([cv_auc(X, y, g, lr, s) for s in range(2)])
null = []
for _ in range(60):
    perm = pd.Series(rng.permutation(pl.values), index=pl.index)
    yp = tr.participant_id.map(perm).values
    null.append(cv_auc(X, yp, g, lr, 0))
null = np.array(null)
print(f"Permutation check (all_audit logreg, 60 participant-level label shuffles): observed {obs:.3f}, "
      f"null mean {null.mean():.3f} sd {null.std():.3f}, p={(1 + (null >= obs).sum()) / 61:.3f}")

print("\n=== 5. STRATA (audit.csv tertiles) ===")
for tcol in ["duration_tertile", "luminance_tertile", "snr_tertile"]:
    if tcol not in df.columns:
        print(tcol, "MISSING")
        continue
    t1 = pd.crosstab(tr[tcol], tr.cohort)
    t2 = te[tcol].value_counts().rename("test")
    tab = t1.join(t2, how="outer").fillna(0).astype(int)
    try:
        p = stats.chi2_contingency(t1.values)[1]
    except Exception:
        p = np.nan
    print(f"\n{tcol} (cohort x tertile chi2 p={p:.3g}; session-level):")
    print(tab.to_string())
    print("  value ranges of the underlying variable per tertile (train+test, min..max):")
    base = {"duration_tertile": "duration_s", "luminance_tertile": "mean_luminance", "snr_tertile": "audio_snr_db"}[tcol]
    print(df.groupby(tcol)[base].agg(["min", "max"]).round(2).to_string())
print("\nINVENTORY DONE; tables in", AUDIT_DIR)

Writing outputs/claude/src/audit/inventory.py


In [14]:
# CELL from the model
apd.run("python outputs/claude/src/audit/inventory.py")

=== 1. INVENTORY ===
audit.csv columns: ['session_id', 'path', 'duration_s', 'width', 'height', 'fps', 'audio_snr_db', 'speech_fraction', 'audio_rms_db', 'error', 'duration_tertile', 'snr_tertile', 'sid']
files by split/ext:
 split  ext 
test   .mp4     62
train  .mov      1
       .mp4    145
label rows: 146 | unique sid in labels: 146 | duplicate stems among all files: 0
labels without file: 0 | train files without label: 0 | audit rows without file: 0 | files without audit row: 0
folder-vs-csv cohort disagreements: 80

Train sessions per cohort:
 cohort
Elevated    80
Control     66
Test sessions (files): 62
Participants per cohort:
 cohort
Control     66
Elevated    80
Participants appearing in both cohorts: 0 of 146
Sessions per participant (count of participants by n sessions):
 1    146
Sessions per participant by cohort (mean, max):
           mean  max
cohort             
Control    1.0    1
Elevated   1.0    1

Merged: train rows 146, test rows 62
Resolution (top 8) train:
 r

1

Use Opus 5.5 with Claude Pro- limit is reached

In [15]:
%%writefile outputs/claude/src/audit/common.py
import os
import numpy as np
import pandas as pd

VID_EXT = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".mpg", ".mpeg", ".wmv", ".flv")
AUDIT_DIR = "outputs/claude/cache/audit"
# train_labels.csv uses "Elevated"; the folder is "Experimental". Both denote the elevated-risk cohort.
COHORT_MAP = {"elevated": "Experimental", "experimental": "Experimental", "control": "Control"}


def norm_id(x):
    return os.path.splitext(os.path.basename(str(x)))[0]


def list_videos(root):
    out = []
    for d, _, fs in os.walk(root):
        for f in sorted(fs):
            if f.lower().endswith(VID_EXT):
                out.append(os.path.join(d, f))
    return sorted(out)


def load_all():
    """Returns (labels, audit, files). All keyed by 'sid' = filename stem. Cohort in {Experimental, Control}."""
    lab = pd.read_csv("data/train/train_labels.csv")
    lab["sid"] = lab["session_id"].map(norm_id)
    lab["cohort_raw"] = lab["cohort"].astype(str).str.strip()
    lab["cohort"] = lab["cohort_raw"].str.lower().map(COHORT_MAP)
    if lab["cohort"].isna().any():
        print("WARNING unmapped cohort labels:", lab.loc[lab.cohort.isna(), "cohort_raw"].unique().tolist())
    aud = pd.read_csv("data/audit.csv")
    key = next((c for c in ["session_id", "video", "filename", "file", "path", "video_path"]
                if c in aud.columns), aud.columns[0])
    aud["sid"] = aud[key].map(norm_id)
    files = list_videos("data/train") + list_videos("data/test")
    ft = pd.DataFrame({"path": files})
    ft["sid"] = ft["path"].map(norm_id)
    ft["split"] = np.where(ft["path"].str.startswith("data/train"), "train", "test")
    ft["ext"] = ft["path"].map(lambda p: os.path.splitext(p)[1].lower())
    ft["folder_cohort"] = ft["path"].map(
        lambda p: "Experimental" if "/Experimental/" in p else ("Control" if "/Control/" in p else ""))
    return lab, aud, ft

Overwriting outputs/claude/src/audit/common.py


In [16]:
%%writefile outputs/claude/src/audit/luminance.py
"""Label-blind mean luminance per video (audit.csv lacks it). 12 evenly spaced frames, grayscale mean/std, cached."""
import os
import numpy as np
import pandas as pd
import cv2
from common import load_all, AUDIT_DIR

OUT = f"{AUDIT_DIR}/luminance.csv"
os.makedirs(AUDIT_DIR, exist_ok=True)
lab, aud, ft = load_all()
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=["sid"])
rows = done.to_dict("records")
have = set(done.sid.astype(str))
todo = ft[~ft.sid.isin(have)]
print(f"videos {len(ft)} | cached {len(have)} | to do {len(todo)}")
for i, (_, r) in enumerate(todo.iterrows(), 1):
    d = dict(sid=r.sid, lum_mean=np.nan, lum_std=np.nan, lum_frames=0)
    try:
        cap = cv2.VideoCapture(r.path)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        m, s = [], []
        if n > 0:
            for k in np.linspace(int(0.03 * n), int(0.97 * n) - 1, 12).astype(int):
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(k))
                ok, fr = cap.read()
                if ok and fr is not None:
                    g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY).astype(np.float32)
                    m.append(g.mean())
                    s.append(g.std())
        cap.release()
        if m:
            d.update(lum_mean=float(np.mean(m)), lum_std=float(np.mean(s)), lum_frames=len(m))
    except Exception:
        pass
    rows.append(d)
    if i % 20 == 0:
        pd.DataFrame(rows).to_csv(OUT, index=False)
        print(f"  {i}/{len(todo)}", flush=True)
L = pd.DataFrame(rows)
L.to_csv(OUT, index=False)
print("lum_mean [min,q1,med,q3,max]:", L.lum_mean.quantile([0, .25, .5, .75, 1]).round(1).tolist(),
      "| videos with 0 frames read:", int((L.lum_frames == 0).sum()))
print("LUMINANCE DONE")

Writing outputs/claude/src/audit/luminance.py


In [17]:
apd.run("python outputs/claude/src/audit/inventory.py")

=== 1. INVENTORY ===
audit.csv columns: ['session_id', 'path', 'duration_s', 'width', 'height', 'fps', 'audio_snr_db', 'speech_fraction', 'audio_rms_db', 'error', 'duration_tertile', 'snr_tertile', 'sid']
files by split/ext:
 split  ext 
test   .mp4     62
train  .mov      1
       .mp4    145
label rows: 146 | unique sid in labels: 146 | duplicate stems among all files: 0
labels without file: 0 | train files without label: 0 | audit rows without file: 0 | files without audit row: 0
folder-vs-csv cohort disagreements: 0

Train sessions per cohort:
 cohort
Experimental    80
Control         66
Test sessions (files): 62
Participants per cohort:
 cohort
Control         66
Experimental    80
Participants appearing in both cohorts: 0 of 146
Sessions per participant (count of participants by n sessions):
 1    146
Sessions per participant by cohort (mean, max):
               mean  max
cohort                 
Control        1.0    1
Experimental   1.0    1

Merged: train rows 146, test rows 

0

In [ ]:
apd.run("python outputs/claude/src/audit/luminance.py")

## Before hand-off

In [ ]:
#@title Check outputs
apd.check_contract()
apd.ls()